# ETH Demo Notebook: 5 Spin-Chain Models

This notebook demonstrates ETH diagonal elements for:

1. Transverse-field Ising chain (TFIM)
2. XXZ chain
3. Disordered Heisenberg chain
4. Random Pauli-string Hamiltonian
5. A custom model built directly with `eth_spin_chain.py` API

For each model, we plot diagonal ETH matrix elements $\langle E_i|O|E_i\rangle$ versus energy $E_i$.

## Setup

In Google Colab, connect to a **CPU** runtime and choose **Runtime > Run all**.
The next cell installs this package and its dependencies from the public
repository's `main` branch. No GPU or Google Drive connection is needed.

For local Jupyter use, first run `python3 -m pip install -e .` from the repository
root and select a kernel using that Python environment. The installation cell
below runs only in Colab.

The default systems have `L=6` or `L=8`. After running the notebook once, try
changing the coupling constants, random seed, or observable in a model cell,
then rerun that cell. Increasing `L` raises the computation and memory cost
exponentially.

In [ ]:
try:
    import google.colab
except ImportError:
    pass  # For local use, install the project before opening this notebook.
else:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "git+https://github.com/osamufukushima/ETH-spin-chain.git@main",
    ])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from eth_spin_chain import (
    DiagonalizedHamiltonian,
    EnergyWindow,
    ObservableSpec,
    ObservableTerm,
    InteractionTerm,
    HamiltonianSpec,
    sample_interaction_terms,
    sampled_interaction_hamiltonian,
    all_site_product_observable,
    eth_elements,
)
from standard_models import (
    transverse_field_ising_chain,
    xxz_chain,
    disordered_heisenberg_chain,
    random_pauli_hamiltonian,
)

plt.rcParams['figure.figsize'] = (6.5, 4.2)
plt.rcParams['axes.grid'] = True

def plot_eth_diagonal_for_model(title, ham, obs, window=None, color='tab:blue'):
    prepared = DiagonalizedHamiltonian.from_spec(ham)
    if window is None:
        result = eth_elements(prepared, obs)
    else:
        result = eth_elements(prepared, obs, window=window)

    fig, ax = plt.subplots()
    ax.scatter(result.energies, np.real(result.diagonal), s=14, alpha=0.65, c=color, edgecolors='none')
    ax.set_title(title)
    ax.set_xlabel('Energy $E_i$')
    ax.set_ylabel('Diagonal ETH element $\\langle E_i|O|E_i\\rangle$')
    plt.show()

    return prepared, result

## 1) Transverse-Field Ising Chain (TFIM)

Hamiltonian:
$$
H_{\mathrm{TFIM}} = -j \sum_i Z_i Z_{i+1} - h \sum_i X_i - g \sum_i Z_i
$$

Observable:
$$
O = Z_0
$$

In [ ]:
L = 8
ham_tfim = transverse_field_ising_chain(L=L, j=1.0, h=1.05, g=0.2, boundary='periodic')
obs_tfim = ObservableSpec([ObservableTerm(sites=[0], ops='Z')])
_tfim_prepared, _tfim_res = plot_eth_diagonal_for_model(
    'TFIM: ETH diagonal elements',
    ham_tfim,
    obs_tfim,
    window=EnergyWindow(center=0.0, width=2.0),
    color='tab:green',
)

## 2) XXZ Chain

Hamiltonian:
$$
H_{\mathrm{XXZ}} = j_{xy} \sum_i (X_iX_{i+1} + Y_iY_{i+1}) + j_z \sum_i Z_iZ_{i+1}
$$

Observable:
$$
O = Z_0 Z_1
$$

In [ ]:
ham_xxz = xxz_chain(L=L, jxy=1.0, jz=0.7, boundary='periodic')
obs_xxz = ObservableSpec([ObservableTerm(sites=[0, 1], ops='ZZ')])
_xxz_prepared, _xxz_res = plot_eth_diagonal_for_model(
    'XXZ: ETH diagonal elements',
    ham_xxz,
    obs_xxz,
    window=EnergyWindow(center=0.0, width=2.0),
    color='tab:blue',
)

## 3) Disordered Heisenberg Chain

Hamiltonian:
$$
H_{\mathrm{dis.Heis}} = \sum_i J_i^{x} X_iX_{i+1} + \sum_i J_i^{y} Y_iY_{i+1} + \sum_i J_i^{z} Z_iZ_{i+1} + \sum_i (h_i^x X_i + h_i^y Y_i + h_i^z Z_i),
$$
with weak disorder ranges for each channel.

Observable:
$$
O = Z_0
$$

In [ ]:
ham_dis = disordered_heisenberg_chain(
    L=L,
    jx=(0.9, 1.1),
    jy=(0.9, 1.1),
    jz=(0.9, 1.1),
    hx=(-0.05, 0.05),
    hy=(-0.05, 0.05),
    hz=(-0.05, 0.05),
    seed=123,
)
obs_dis = ObservableSpec([ObservableTerm(sites=[0], ops='Z')])
_dis_prepared, _dis_res = plot_eth_diagonal_for_model(
    'Disordered Heisenberg: ETH diagonal elements',
    ham_dis,
    obs_dis,
    window=EnergyWindow(center=0.0, width=2.0),
    color='tab:red',
)

## 4) Random Pauli-String Hamiltonian

Hamiltonian:
$$
H_{\mathrm{rand}} = \sum_{k=1}^{n_{\mathrm{terms}}} c_k P_k,
$$
where $P_k$ are sampled Pauli strings and $c_k$ are sampled coefficients.

Observable:
$$
O = X_0 X_1
$$

In [ ]:
L_rand = 6
ham_rand = random_pauli_hamiltonian(
    L=L_rand,
    term_width=3,
    n_terms=12,
    coeff_min=-1.0,
    coeff_max=1.0,
    seed=7,
)
obs_rand = ObservableSpec([ObservableTerm(sites=[0, 1], ops='XX')])
_rand_prepared, _rand_res = plot_eth_diagonal_for_model(
    'Random Pauli Hamiltonian: ETH diagonal elements',
    ham_rand,
    obs_rand,
    window=None,
    color='tab:purple',
)

## 5) Custom Model Built Directly with `eth_spin_chain.py` API

This section uses API building blocks not used in the four standard demos:

- `sampled_interaction_hamiltonian(...)` to sample translated `XY` couplings
- direct manual term injection via `InteractionTerm`
- `all_site_product_observable(...)` for a nonlocal observable

Model:
$$
H = \sum_i J_i X_iY_{i+1} + g\sum_i Z_iZ_{i+1} + \lambda \sum_i X_i + \kappa X_0Y_1Z_2
$$
with sampled $J_i \in [0.8, 1.2]$.

Observable (nonlocal):
$$
O = \prod_{i=0}^{L-1} Z_i
$$

In [ ]:
L_custom = 8
base_xy = sampled_interaction_hamiltonian(
    L=L_custom,
    ops='XY',
    j_min=0.8,
    j_max=1.2,
    boundary='periodic',
    seed=2026,
)

zz_terms = sample_interaction_terms(
    L=L_custom,
    ops='ZZ',
    j_min=0.6,
    j_max=0.6,
    boundary='periodic',
)

x_field_terms = [InteractionTerm(sites=[i], ops='X', coeff=0.25) for i in range(L_custom)]
three_body = InteractionTerm(sites=[0, 1, 2], ops='XYZ', coeff=0.15)

ham_custom = HamiltonianSpec(
    L=L_custom,
    boundary='periodic',
    terms=[*base_xy.terms, *zz_terms, *x_field_terms, three_body],
)

obs_custom = all_site_product_observable(L=L_custom, ops='Z')
_custom_prepared, _custom_res = plot_eth_diagonal_for_model(
    'Custom API-built model: ETH diagonal elements',
    ham_custom,
    obs_custom,
    window=EnergyWindow(center=0.0, width=2.0),
    color='tab:orange',
)

## Notes

- Exact diagonalization scales exponentially with system size ($2^L$).
- For interactive use, start with small $L$ values (for example, $L=6$ to $L=10$).
- To compare models fairly, keep observable definitions and energy-window widths consistent.